# LMA Evaluation — Model H (Hindi) & Model L (Nepali)

Runs everything in assignment §2.3 ("Language Modeling and Attention Evaluation") against
the already-pretrained checkpoints on the Hugging Face Hub: test-set perplexity/BPB,
generation quality (BLEU-4/chrF++/ROUGE-L + diversity diagnostics) at greedy + 3
temperatures, and attention heatmaps/entropy/mean-distance — for both languages.

This is forward-pass-only work against a finished checkpoint (no training), so unlike
`kaggle_pretrain.ipynb` it needs neither GPU T4x2 nor multi-session resume — one GPU, one
run, done in minutes.

**Before running:**

1. **Add these two datasets** (Add Input):

   | Dataset | Used for |
   | --- | --- |
   | `sohailmemon/lma-individual` | `<lang>/test.txt` (raw text, for BPB's byte count and generation prefixes/references) and `tokenized/<lang>/test.bin` (if present, else this notebook tokenizes it) |
   | `sohailmemon/lma-individual-files` | `<lang>/tokenizer/*.model` and `scripts/{training,evaluation}/*.py` |

2. **Add the `HF_TOKEN` Kaggle secret** (Add-ons -> Secrets) — needed to download the
   private checkpoint repos (`sohailmemon/lma-hindi-gpt`, `sohailmemon/lma-nepali-gpt`).
3. GPU accelerator: any single GPU is enough (T4 or P100).

All results (JSON + PNGs) are written under `/kaggle/working/results/<lang>/` — download
them from the notebook's output after running and bring them back into the repo/report,
the same way the pretraining logs were.


## 0. Setup

In [ ]:
!pip install -q sentencepiece huggingface_hub sacrebleu rouge-score
!apt-get update -qq > /dev/null && apt-get install -y fonts-noto -qq > /dev/null


In [ ]:
import shutil
from pathlib import Path

FILES_DS = Path("/kaggle/input/datasets/sohailmemon/lma-individual-files")
DATA_DS = Path("/kaggle/input/datasets/sohailmemon/lma-individual")
WORK = Path("/kaggle/working")

if (WORK / "scripts").exists():
    shutil.rmtree(WORK / "scripts")
shutil.copytree(FILES_DS / "scripts", WORK / "scripts")

import sys
sys.path.insert(0, str(WORK))

print("scripts copied to:", WORK / "scripts")


In [ ]:
import os

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("loaded HF_TOKEN from Kaggle secrets")
except Exception as e:
    print(f"could not load Kaggle secrets ({e}) — relying on an already-set HF_TOKEN")

assert os.environ.get("HF_TOKEN"), "add the HF_TOKEN Kaggle secret (Add-ons -> Secrets)"


## 1. Download checkpoints

In [ ]:
from huggingface_hub import hf_hub_download

LANGS = ["hindi", "nepali"]
HF_USERNAME = "sohailtsm"
HF_REPOS = {lang: f"{HF_USERNAME}/lma-{lang}-gpt" for lang in LANGS}

checkpoints = {}
for lang in LANGS:
    checkpoints[lang] = hf_hub_download(HF_REPOS[lang], "best.pt")
    print(f"[{lang}] {checkpoints[lang]}")


## 2. Ensure tokenized test data exists

Same pre-tokenization check as `kaggle_pretrain.ipynb`: uses `tokenized/<lang>/test.bin`
from the dataset if present, otherwise tokenizes `test.txt` on the spot.


In [ ]:
import subprocess

BIN_DIR = {}
for lang in LANGS:
    pretokenized = DATA_DS / "tokenized" / lang
    if (pretokenized / "test.bin").exists():
        print(f"[{lang}] found pre-tokenized test.bin in {pretokenized}")
        BIN_DIR[lang] = pretokenized
        continue

    out_dir = WORK / lang / "data"
    out_dir.mkdir(parents=True, exist_ok=True)
    cmd = [
        "python", "-u", "-m", "scripts.training.prepare_bin",
        "--lang", lang,
        "--data-dir", str(DATA_DS / lang),
        "--tokenizer", str(FILES_DS / lang / "tokenizer" / f"{lang}_unigram_16k.model"),
        "--out-dir", str(out_dir),
    ]
    print(f"=== tokenizing {lang} ===")
    subprocess.run(cmd, cwd=WORK, check=True)
    BIN_DIR[lang] = out_dir


## 3. Test-set perplexity and bits-per-byte


In [ ]:
RESULTS_DIR = WORK / "results"

for lang in LANGS:
    cmd = [
        "python", "-u", "-m", "scripts.evaluation.eval_lm",
        "--lang", lang,
        "--checkpoint", checkpoints[lang],
        "--data-dir", str(BIN_DIR[lang]),
        "--test-text", str(DATA_DS / lang / "test.txt"),
        "--out", str(RESULTS_DIR / lang / f"{lang}_lm_metrics.json"),
    ]
    subprocess.run(cmd, cwd=WORK, check=True)


## 4. Generation quality: BLEU-4, chrF++, ROUGE-L, diversity

Greedy decoding plus temperatures 0.5 / 1.0 / 1.5, per assignment §2.3.


In [ ]:
for lang in LANGS:
    cmd = [
        "python", "-u", "-m", "scripts.evaluation.generate_eval",
        "--lang", lang,
        "--checkpoint", checkpoints[lang],
        "--tokenizer", str(FILES_DS / lang / "tokenizer" / f"{lang}_unigram_16k.model"),
        "--test-text", str(DATA_DS / lang / "test.txt"),
        "--out", str(RESULTS_DIR / lang / f"{lang}_generation.json"),
    ]
    subprocess.run(cmd, cwd=WORK, check=True)


## 5. Attention heatmaps, entropy, mean distance


In [ ]:
for lang in LANGS:
    cmd = [
        "python", "-u", "-m", "scripts.evaluation.attention_analysis",
        "--lang", lang,
        "--checkpoint", checkpoints[lang],
        "--tokenizer", str(FILES_DS / lang / "tokenizer" / f"{lang}_unigram_16k.model"),
        "--test-text", str(DATA_DS / lang / "test.txt"),
        "--out-dir", str(RESULTS_DIR / lang / "attention"),
    ]
    subprocess.run(cmd, cwd=WORK, check=True)


## 6. Summary

In [ ]:
import json

for lang in LANGS:
    print(f"=== {lang} ===")
    lm = json.loads((RESULTS_DIR / lang / f"{lang}_lm_metrics.json").read_text())
    print(f"  test loss: {lm['test_loss_nats']:.4f}  perplexity: {lm['perplexity']:.2f}  "
          f"BPB: {lm['bpb']:.4f}")

    gen = json.loads((RESULTS_DIR / lang / f"{lang}_generation.json").read_text())
    for name, m in gen["metrics"].items():
        print(f"  [{name}] BLEU-4={m['bleu4']:.2f} chrF++={m['chrf++']:.2f} "
              f"ROUGE-L={m['rougeL_f1']:.3f} repetition={m['repetition_rate_4gram']:.3f} "
              f"distinct-1={m['distinct_1']:.3f} distinct-2={m['distinct_2']:.3f}")

print("\nAll files (download these and bring them into the repo/report):")
for p in sorted(RESULTS_DIR.rglob("*")):
    if p.is_file():
        print(" ", p.relative_to(WORK))
